In [5]:
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader, Dataset

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
# import seaborn as sns

from tqdm.notebook import trange, tqdm


# plt.style.use('seaborn-v0_8-darkgrid')
# sns.set_palette('husl')
torch.manual_seed(42)

In [11]:
# Hyperparamters
learning_rate = 1e-4
# n_epochs = 20
n_epochs = 2 # just to get things setup
batch_size = 32


In [6]:
class CSVDataset(Dataset):
    
    def __init__(self, csv_file):

        df = pd.read_csv(csv_file)
        
        X = df["sequence_content"].values
        y = df["TE_Order"].values
    
        self.X = torch.tensor(X, dtype=torch.string)
        self.y = torch.tensor(y, dtype=torch.string)

    def __len__(self):
        return len(self.y)

training_data = CSVDataset(csv_file="Preprocessed/train.csv")
testing_data = CSVDataset(csv_file="Preprocessed/test.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'Preprocessed/train.csv'

In [ ]:
data_loader_train = DataLoader(training_data, batch_size=batch_size, shuffle=True)

In [6]:
# contructor defines the layers
# forward describes how the data flows through the layers


class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size
        )

        # classification layer (fully connected)
        self.fc_out = nn.Linear(hidden_size, output_size)
    
    def forward(self, input_seq):

        # output contains hidden state from each timestamp, from final lstm layer
        # hidden_out is the final hidden state for each LSTM layer -> use -1 to get last layer
        output, (hidden_out, mem_out) = self.lstm(input_seq)

        final_hidden = hidden_out[-1]
        
        return self.fc_out(final_hidden)
        
        


In [7]:
device = torch.device(0 if torch.cuda.is_available() else 'cpu') # update for macOS specific

In [12]:
input_size = 4
hidden_size = 128
output_size = 10 # this needs to be double checked

# Create model
lstm_classifier = LSTM(input_size, hidden_size, output_size)

# Initialize the optimizer with the above parameters
optimizer = optim.Adam(lstm_classifier.parameters(), lr=learning_rate)

# Define the loss function
loss_fn = nn.CrossEntropyLoss()


In [13]:
num_model_params = 0

for param in lstm_classifier.parameters():
    num_model_params += param.flatten().shape[0]

print(f"The model has {num_model_params} parameters")

The model has 69898 parameters


In [14]:
training_loss_logger = []
test_loss_logger = []

training_acc_logger = []
testing_acc_logger = []

In [ ]:
train_acc = 0
test_acc = 0

for epoch in n_epochs:

    print (epoch)

    # set model to training mode
    lstm_classifier.train()
    steps = 0

    # Go through training data
    for labels, sequences in training_data:

        batch_size = label.shape[0]

        # do something like one_hot encoding for DNA sequences
        
        # Initialize hidden and memory states
        hidden = torch.zeros(num_layers, batch_size, hidden_size, device = device)
        memory = torch.zeros(num_layers, batch_size, hidden_size, device = device)

        # forward pass
        pred, hidden, memory = lstm_classifier(sequences)
    
        # loss calculation

        # backpropagation and optimization

        # append training loss to logger

        # calculate training accuracy
